# Logistic Regression Baseline for Elliptic Bitcoin Dataset

This notebook implements a logistic regression baseline for the Elliptic Bitcoin transaction classification task. It reads `elliptic_txs_features.csv` and `elliptic_txs_classes.csv`, merges them by `txId`, and keeps only labeled samples where `class ∈ {1, 2}`, mapping `class == "1"` to illicit (1) and `class == "2"` to licit (0). The notebook heuristically detects a time-step column and, if available, uses `StratifiedGroupKFold` to reduce temporal leakage; otherwise it falls back to `StratifiedKFold`. Features are standardized and expanded with low-degree polynomial terms via a `Pipeline` (`StandardScaler` → `PolynomialFeatures(degree=2)` → `LogisticRegression(class_weight="balanced")`), and the model is evaluated with 5-fold cross-validation, printing detailed classification reports at a decision threshold of 0.5 for each fold.


In [2]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# splitters
from sklearn.model_selection import StratifiedKFold
try:
    from sklearn.model_selection import StratifiedGroupKFold
    HAS_SGK = True
except Exception:
    HAS_SGK = False


# =========================
# 0) 路径 & 参数
# =========================
classes_path  = "elliptic_txs_classes.csv"
features_path = "elliptic_txs_features.csv"

N_SPLITS = 5
RANDOM_STATE = 42
THRESHOLD = 0.5

# Polynomial degree（代数统计味道：低次数多项式判别）
POLY_DEGREE = 2

# =========================
# 1) 读入数据
# =========================
df_y = pd.read_csv(classes_path)
df_x = pd.read_csv(features_path, header=None)

# features 第一列一般是 txId
df_x = df_x.rename(columns={0: "txId"})

# 统一 txId dtype，避免 merge 失败
df_x["txId"] = df_x["txId"].astype(str)
df_y["txId"] = df_y["txId"].astype(str)

# 统一 class dtype，避免 isin 失效
df_y["class"] = df_y["class"].astype(str).str.strip()

df = df_x.merge(df_y, on="txId", how="inner")

print("After merge:", df.shape)
print("class unique:", df["class"].unique()[:10], " ... (showing first up to 10)")

# =========================
# 2) 只保留已标注：class in {"1","2"}
# =========================
df_labeled = df[df["class"].isin(["1", "2"])].copy()
if len(df_labeled) == 0:
    raise ValueError(
        "No labeled samples found after filtering class in {'1','2'}. "
        "Please print df['class'].unique() and check the label encoding."
    )

# illicit=1, licit=0
df_labeled["y"] = (df_labeled["class"] == "1").astype(int)

print("Labeled samples:", len(df_labeled))
print("y counts:\n", df_labeled["y"].value_counts())

# =========================
# 3) 识别 time_step 列（用于 group split，减少时间泄漏）
#    Elliptic 常见第2列是 time step（col=1）
# =========================
# 候选：原 features 的列 1
time_col_candidate = 1 if 1 in df_x.columns else None

def looks_like_time_step(series: pd.Series) -> bool:
    # time step 常见：整数、小范围(例如 1..49)，唯一值不太多
    s = pd.to_numeric(series, errors="coerce")
    if s.isna().mean() > 0.5:
        return False
    uniq = s.nunique(dropna=True)
    if uniq < 2:
        return False
    # 太多唯一值（接近样本数）就不像 time step
    if uniq > min(200, int(0.5 * len(series))):
        return False
    return True

use_group_split = False
group_col = None

if time_col_candidate is not None and looks_like_time_step(df_labeled[time_col_candidate]):
    use_group_split = HAS_SGK
    group_col = time_col_candidate
else:
    # 如果第2列不像 time_step，就尝试在前几列里找一个像 time step 的
    for c in list(df_x.columns)[:10]:
        if c == "txId":
            continue
        if looks_like_time_step(df_labeled[c]):
            use_group_split = HAS_SGK
            group_col = c
            break

if use_group_split:
    print(f"Using StratifiedGroupKFold with group column = {group_col} (likely time_step).")
else:
    if HAS_SGK:
        print("StratifiedGroupKFold available, but no reliable time_step column detected. Falling back to StratifiedKFold.")
    else:
        print("StratifiedGroupKFold not available. Falling back to StratifiedKFold (may leak across time).")

# =========================
# 4) 构造 X, y
# =========================
# 特征列：除了 txId 和 class/y 和 group_col（如果你不想把 time_step 当特征）
drop_cols = {"txId", "class", "y"}
if group_col is not None:
    drop_cols.add(group_col)

feature_cols = [c for c in df_labeled.columns if c not in drop_cols]

# 全部转成数值（有些列可能是字符串，强制转）
X_df = df_labeled[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0)
X = X_df.to_numpy(dtype=float)
y = df_labeled["y"].to_numpy(dtype=int)

groups = None
if use_group_split and group_col is not None:
    groups = pd.to_numeric(df_labeled[group_col], errors="coerce").fillna(-1).astype(int).to_numpy()

print("X shape:", X.shape, "y shape:", y.shape)

# =========================
# 5) 模型：Standardize + PolynomialFeatures + LogisticRegression
# =========================
model = Pipeline([
    ("scaler", StandardScaler()),
    ("poly", PolynomialFeatures(degree=POLY_DEGREE, include_bias=False)),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        solver="lbfgs",
        n_jobs=None
    ))
])

# =========================
# 6) 5-fold 训练 & 测试报告（threshold=0.5）
# =========================
if use_group_split:
    splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    split_iter = splitter.split(X, y, groups=groups)
else:
    splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    split_iter = splitter.split(X, y)

for fold, (train_idx, test_idx) in enumerate(split_iter, start=1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model.fit(X_train, y_train)

    proba = model.predict_proba(X_test)[:, 1]
    y_pred = (proba >= THRESHOLD).astype(int)

    # sanity prints
    pos_train = int(y_train.sum()); neg_train = int((1-y_train).sum())
    pos_test  = int(y_test.sum());  neg_test  = int((1-y_test).sum())

    print(f"\n--- Fold {fold} ---")
    print(f"Train: n={len(train_idx)} (illicit={pos_train}, licit={neg_train}) | "
          f"Test: n={len(test_idx)} (illicit={pos_test}, licit={neg_test})")
    print(f"--- TEST report @ threshold={THRESHOLD} ---")
    print(classification_report(
        y_test, y_pred,
        target_names=["licit(0)", "illicit(1)"],
        digits=2
    ))

After merge: (203769, 168)
class unique: ['unknown' '2' '1']  ... (showing first up to 10)
Labeled samples: 46564
y counts:
 y
0    42019
1     4545
Name: count, dtype: int64
Using StratifiedGroupKFold with group column = 1 (likely time_step).
X shape: (46564, 165) y shape: (46564,)

--- Fold 1 ---
Train: n=36763 (illicit=3880, licit=32883) | Test: n=9801 (illicit=665, licit=9136)
--- TEST report @ threshold=0.5 ---
              precision    recall  f1-score   support

    licit(0)       0.99      0.97      0.98      9136
  illicit(1)       0.65      0.87      0.74       665

    accuracy                           0.96      9801
   macro avg       0.82      0.92      0.86      9801
weighted avg       0.97      0.96      0.96      9801


--- Fold 2 ---
Train: n=35681 (illicit=3459, licit=32222) | Test: n=10883 (illicit=1086, licit=9797)
--- TEST report @ threshold=0.5 ---
              precision    recall  f1-score   support

    licit(0)       0.99      0.98      0.98      9797
  illi